# 11 - ניתוח רשת הרכבות בלבד

קובץ ה-GTFS הארצי נשלט על ידי אוטובוסים: עשרות אלפי קווים ומיליוני רשומות של זמני עצירה.
הרכבת הכבדה (`route_type = 2`) היא תת-רשת זעירה בתוכו - כ-67 תחנות פעילות ו-99 קשתות שירות
בלתי מכוונות - והגודל הקטן הזה הוא הזדמנות. כל מה שהיה חייב להיות מדגם או קירוב ברשת
האוטובוסים (בפרט betweenness) ניתן לחישוב **מדויק** כאן, וכל סגירה של תחנה בודדת ניתנת
לסימולציה בכוח גס. מחברת זו מחלצת את שכבת הרכבת מהקובץ הגולמי, בונה את הגרף, מודדת
מרכזיות, מאתרת את נקודות החיתוך (articulation points) והגשרים (bridges), ומשווה סגירות תחנה
ממוקדות מול כשלים אקראיים.

**שאלת המחקר.** אילו תחנות רכבת הן קריטיות מבחינה מבנית, ועד כמה תקיפה מכוונת על רשת הרכבת
חמורה יותר ממספר זהה של כשלים אקראיים?

**קלט**
- `israel-public-transportation/routes.txt`, `trips.txt`, `stops.txt`, `calendar.txt` (מנוהלים ב-git)
- `israel-public-transportation/stop_times.txt` (816 MB, **אינו** מנוהל ב-git - מורד לפי דרישה)

**פלט** (הכול תחת `outputs/nb/11_rail_network_analysis/`)
- `tables/mode_inventory.csv` - מספר קווים/נסיעות לכל אופן תחבורה ב-GTFS, המראה עד כמה הרכבת קטנה
- `tables/rail_station_metrics.csv` - לכל תחנה: degree, PageRank, betweenness מדויק, harmonic, קהילה, סימון articulation
- `tables/top_betweenness_stations.csv`, `tables/top_weighted_degree_stations.csv`
- `tables/rail_articulation_points.csv`, `tables/rail_bridges.csv`
- `tables/single_station_damage.csv` - הפיצול הנגרם מסגירת כל תחנה בנפרד
- `tables/rail_resilience_curves.csv` - עקומות הסרה ממוקדת מול אקראית
- `tables/rail_service_edges.csv` - רשימת קשתות מכוונת עם משקלי רשומות נסיעה
- `summary.json` - ספירות בנייה וסטטיסטיקות של הרשת כולה
- `figures/` - מפת הרשת, עמודות top-betweenness, עמודות נזק מתחנה בודדת, עקומות עמידות, מפת חום של מתאמי מרכזיות

**שלבים קודמים נדרשים:** אין. מחברת זו קוראת ישירות את קובץ ה-GTFS הגולמי והיא עצמאית.

---

## יש לקרוא זאת לפני פירוש כל מספר להלן

**1. זהו גרף של שירות מתוזמן, ולא מפה של תשתית מסילתית פיזית.** קשת קיימת בין שתי תחנות
בכל פעם שרכבת מתוזמנת כלשהי עוצרת באחת ואז בשנייה. שירותי אקספרס מדלגים על תחנות, ולכן הם
יוצרים קשתות בין תחנות ש*אינן* סמוכות פיזית. הגרף מקודד אפוא את *אופן הפעלת השירות*, וזה בדיוק
מה שרצוי כאשר שואלים "מי עדיין מקבל רכבת אם תחנה זו נסגרת", אך אין לקרוא אותו כתרשים תשתית.
גשר (bridge) בגרף זה הוא גשר בלוח הזמנים, ולא בהכרח קו מסילה פיזי יחיד.

**2. משקלי הקשתות סופרים רשומות נסיעה ב-GTFS, ולא ימי שירות.** משקל של 219 פירושו 219
רשומות נסיעה בקובץ החוצות את אותה קשת. GTFS מפצל לוח זמנים לתבניות שירות, ולכן וריאנט הפועל
רק בימי ראשון תורם בדיוק אותו משקל כמו וריאנט יומיומי בעל אותו מספר נסיעות. המשקלים הם
פרוקסי ל*עושר תבניות השירות*, ולא למספר רכבות לשבוע. כל המסקנות המבניות להלן (נקודות חיתוך,
גשרים, פיצול) משתמשות בטופולוגיה **הלא-משוקללת** ואינן מושפעות מכך; רק weighted degree
ו-PageRank יורשים את הסייג הזה.

## אתחול סביבת העבודה

מאתר את המאגר (ומשכפל אותו ב-Google Colab), מתקין רק את החבילות שאכן חסרות, ויוצר את תיקיית
הפלט הראשית. הרצה חוזרת של תא זה בטוחה.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## ספריות וקבועי הניתוח

כל מה ששולט בעלות החישוב או בשחזוריות מרוכז במקום אחד. בגרף הרכבת יש רק 67 צמתים, ולכן אין
צורך לדגום אף מדד מרכזיות - הפרמטר היקר כאן הוא ניסוי מונטה-קרלו של הכשלים האקראיים, המריץ
`RANDOM_TRIALS` ערבובים בלתי תלויים ומעריך מחדש את הקשירות לאחר כל שלב הסרה (500 ניסויים
כפול 16 שלבים אורכים מספר שניות). `SEED` מקבע הן את זיהוי הקהילות בשיטת Louvain והן את סדרי
הכשלים האקראיים, כך שהמחברת משוחזרת במדויק.

In [ ]:
_ensure("networkx", "pandas", "numpy", "matplotlib", "seaborn")

import csv
import json
import random
import time
from collections import Counter, defaultdict

import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

# --- Analysis constants -------------------------------------------------
RAIL_ROUTE_TYPE = "2"   # GTFS route_type 2 = heavy rail (Israel Railways)
SEED = 42               # fixes Louvain communities and random-failure orderings
RANDOM_TRIALS = 500     # Monte-Carlo repetitions for the random-failure baseline
MAX_REMOVALS = 15       # how far the resilience curves run along the x-axis

# --- Stage output folders ----------------------------------------------
STAGE = OUT / "11_rail_network_analysis"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("networkx", nx.__version__, "| pandas", pd.__version__)
print("Stage folder:", STAGE)

## הורדת קובץ זמני העצירה הגולמי

`stop_times.txt` שוקל 816 MB - גדול מכדי להישמר ב-git - ולכן הוא מאוחסן בנפרד ונמשך לפי דרישה.
כל אירועי העצירה במדינה נמצאים בקובץ היחיד הזה (כ-15.7 מיליון שורות); תת-הקבוצה של הרכבת
הדרושה לנו מונה כ-14,600 מהן.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## עיבוד טקסט בעברית

כל שמות התחנות בקובץ הם בעברית. Matplotlib משרטט תווים בסדר לוגי ואינו מיישם את אלגוריתם
הדו-כיווניות (bidirectional) של Unicode, ולכן ללא תיקון זה כל תווית בעברית מוצגת הפוכה.
אנו מתקנים את `Text.set_text` פעם אחת, לפני שרטוט כל איור, כדי שנוכל להעביר מחרוזות עבריות
גולמיות בכל שאר המחברת ולקבל פלט תקין.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

# Because the patch is global, pass RAW Hebrew strings to matplotlib from here on -
# calling fix_he() manually as well would reverse the text twice.

## טעינת מטא-נתוני ה-GTFS ומדידת גודלו של כל אופן תחבורה

ארבע טבלאות ה-GTFS הקטנות נקראות כמחרוזות גולמיות: מזהי GTFS הם קודים אטומים, ואילו נתנו
ל-pandas לנחש טיפוסים מספריים היו נמחקים אפסים מובילים בשקט והחיבורים (joins) היו נשברים.
מלאי אופני התחבורה מחבר לאחר מכן את `trips` אל `routes`, כך שניתן לראות כמה רשומות קווים
וכמה נסיעות מתוזמנות תורם כל אופן תחבורה - זו הראיה לכך שהרכבת היא פלח שולי בקובץ מבחינת
נפח, ולכן זולה לניתוח ממצה.

In [ ]:
def read_gtfs(path):
    """Read a GTFS table as strings. Fails loudly on a missing or placeholder file."""
    if not path.exists() or path.stat().st_size < 100:
        raise FileNotFoundError(
            f"Missing or unhydrated GTFS file: {path}. "
            "Check that the israel-public-transportation folder was cloned in full."
        )
    return pd.read_csv(path, dtype=str, encoding="utf-8-sig")

def route_type_label(route_type):
    """Human-readable name for the GTFS route_type codes used in this feed."""
    return {
        "0": "tram/light rail",
        "2": "rail",
        "3": "bus",
        "5": "cable tram",
        "8": "trolleybus/taxi-coded",
        "715": "demand/other bus",
    }.get(str(route_type), "other")

def mode_inventory(routes, trips):
    """Route records and scheduled trips per mode."""
    route_counts = routes.groupby("route_type").size().rename("route_records")
    trip_counts = (
        trips.merge(routes[["route_id", "route_type"]], on="route_id", how="left")
        .groupby("route_type")
        .size()
        .rename("scheduled_trips")
    )
    inventory = pd.concat([route_counts, trip_counts], axis=1).fillna(0).reset_index()
    inventory["route_type_label"] = inventory["route_type"].map(route_type_label)
    inventory["route_records"] = inventory["route_records"].astype(int)
    inventory["scheduled_trips"] = inventory["scheduled_trips"].astype(int)
    return inventory[
        ["route_type", "route_type_label", "route_records", "scheduled_trips"]
    ]

routes = read_gtfs(DATA / "routes.txt")
trips = read_gtfs(DATA / "trips.txt")
stops = read_gtfs(DATA / "stops.txt")
calendar = read_gtfs(DATA / "calendar.txt")

inventory = mode_inventory(routes, trips)
print(f"routes={len(routes):,}  trips={len(trips):,}  stops={len(stops):,}")
inventory

## בידוד שכבת הרכבת

אנו שומרים את הקווים שבהם `route_type` שווה ל-2 ואת כל הנסיעות השייכות לאחד מהם. קבוצת
ה-`trip_id` המתקבלת היא המסנן המשמש בעת קריאת קובץ זמני העצירה הענק. כמו כן אנו קוראים את
לוח השירות (calendar) של אותן נסיעות, כדי שהדוח יוכל לציין במדויק לאיזה חלון לוח זמנים
מתייחסות המסקנות - קובצי GTFS הם תצלומי מצב, ותוצאה שהופקה מחלון של חודש אחד אינה צריכה
להיות מוצגת כתכונה קבועה של הרשת.

In [ ]:
rail_routes = routes[routes["route_type"] == RAIL_ROUTE_TYPE].copy()
rail_trips = trips[trips["route_id"].isin(rail_routes["route_id"])].copy()
if rail_routes.empty or rail_trips.empty:
    raise RuntimeError("The GTFS feed contains no route_type=2 rail service")

def service_date_range(rail_trips, calendar):
    """Earliest start and latest end date of the calendars used by rail trips."""
    relevant = calendar[calendar["service_id"].isin(rail_trips["service_id"])]
    if relevant.empty:
        return "", ""
    def iso_date(raw):
        raw = str(raw)
        return f"{raw[:4]}-{raw[4:6]}-{raw[6:8]}" if len(raw) == 8 else raw
    return iso_date(relevant["start_date"].min()), iso_date(relevant["end_date"].max())

start_date, end_date = service_date_range(rail_trips, calendar)
rail_trip_ids = set(rail_trips["trip_id"])

print(f"rail route records      : {len(rail_routes):,}")
print(f"unique route long names : {rail_routes['route_long_name'].nunique():,}")
print(f"scheduled rail trips    : {len(rail_trips):,}")
print(f"service window          : {start_date} .. {end_date}")

## קריאה זורמת של שורות הרכבת מתוך `stop_times.txt`

הקובץ מכיל כ-15.7 מיליון אירועי עצירה ואינו נכנס בנוחות לזיכרון כ-DataFrame, ולכן אנו
קוראים אותו שורה-שורה באמצעות מודול `csv` הסטנדרטי ושומרים רק את השורות שבהן `trip_id` הוא
נסיעת רכבת - כמה אלפי שורות שורדות. לכל נסיעה ששרדה אנו אוספים זוגות
`(stop_sequence, stop_id)`, וכן סופרים בכמה פעמים נעצרים בכל תחנה, מדד שימושי לבדיקת שפיות
מול ה-degree בהמשך.

**עלות:** זהו התא האיטי ביותר במחברת - כ-1 עד 3 דקות, בהתאם למהירות הדיסק, שכן יש לפרסר את
כל 15.7 מיליון השורות אף שכמעט כולן נזרקות. אנו ממפים את השורה הראשונה פעם אחת ומשתמשים
ב-`csv.reader` במקום ב-`csv.DictReader`, שהוא מהיר משמעותית בקובץ בגודל כזה ומתנהג באופן זהה.

In [ ]:
def collect_rail_stop_times(stop_times_path, rail_trip_ids):
    """Stream the national stop-times file but retain only the small rail subset.

    Returns (sequences, stop_use_counts, total_rows_seen, rail_rows_kept).
    """
    sequences = defaultdict(list)
    stop_use_counts = Counter()
    all_rows = 0
    rail_rows = 0
    with stop_times_path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        required = {"trip_id", "stop_id", "stop_sequence"}
        if not required.issubset(set(header)):
            raise ValueError(f"stop_times.txt is missing fields: {sorted(required)}")
        i_trip = header.index("trip_id")
        i_stop = header.index("stop_id")
        i_seq = header.index("stop_sequence")
        for row in reader:
            all_rows += 1
            if row[i_trip] not in rail_trip_ids:
                continue
            rail_rows += 1
            try:
                sequence = int(float(row[i_seq]))
            except ValueError:
                continue  # unparsable sequence: the row cannot be ordered, so skip it
            stop_id = row[i_stop]
            sequences[row[i_trip]].append((sequence, stop_id))
            stop_use_counts[stop_id] += 1
    return sequences, stop_use_counts, all_rows, rail_rows

_t0 = time.perf_counter()
sequences, stop_calls, source_rows, rail_rows = collect_rail_stop_times(
    STOP_TIMES, rail_trip_ids
)
print(f"scanned {source_rows:,} stop-time rows in {time.perf_counter() - _t0:.1f}s")
print(f"kept {rail_rows:,} rail rows across {len(sequences):,} trips")
print(f"rail share of the feed: {100 * rail_rows / max(source_rows, 1):.4f}%")

## בניית גרף הרכבת

הגדרת הגרף תואמת במכוון את ניתוח רשת האוטובוסים המרכזי, כדי שהשניים יהיו ברי-השוואה:

- **צומת** = תחנת רכבת ישראל פעילה
- **קשת מכוונת** `u -> v` = רכבת מתוזמנת כלשהי עוצרת ב-`u` ומיד לאחר מכן ב-`v`
- **משקל** = מספר רשומות הנסיעה המתוזמנות המשתמשות בקשת שבין שתי התחנות

רשימת התחנות של כל נסיעה ממוינת לפי `stop_sequence` (GTFS אינו מבטיח את סדר השורות בקובץ),
וזוגות עוקבים הופכים לקשתות; לולאות עצמיות הנובעות משורות כפולות מוסרות. התצוגה הבלתי מכוונת
מסכמת את המשקלים של שני הכיוונים, והיא זו שעליה מתבסס כל הניתוח המבני, שכן תחנה סגורה חוסמת
נסיעה בשני הכיוונים.

יש לזכור את שני הסייגים שבראש המחברת: עוקב *בלוח הזמנים* אינו זהה לסמוך *על המסילה*, והמשקל
סופר רשומות נסיעה ולא ימי שירות.

In [ ]:
def build_graphs(sequences):
    """Turn per-trip stop sequences into a weighted directed graph and its undirected view."""
    edge_counts = Counter()
    nodes = set()
    for trip_sequence in sequences.values():
        ordered = [stop for _, stop in sorted(trip_sequence)]
        nodes.update(ordered)
        for source, target in zip(ordered, ordered[1:]):
            if source != target:  # ignore duplicated rows for the same station
                edge_counts[(source, target)] += 1

    directed = nx.DiGraph()
    directed.add_nodes_from(nodes)
    for (source, target), weight in edge_counts.items():
        directed.add_edge(source, target, weight=int(weight))

    undirected = nx.Graph()
    undirected.add_nodes_from(nodes)
    for source, target, data in directed.edges(data=True):
        weight = int(data["weight"])
        if undirected.has_edge(source, target):
            undirected[source][target]["weight"] += weight
        else:
            undirected.add_edge(source, target, weight=weight)
    return directed, undirected

directed, undirected = build_graphs(sequences)
print(f"stations               : {undirected.number_of_nodes()}")
print(f"directed service edges : {directed.number_of_edges()}")
print(f"undirected service edges: {undirected.number_of_edges()}")
print(f"connected components   : {nx.number_connected_components(undirected)}")

## צירוף שמות התחנות והקואורדינטות

עד כה הגרף מכיר רק ערכי `stop_id` אטומים. כאן אנו מחברים לצמתים את הקואורדינטות ואת השמות
בעברית מתוך `stops.txt`, כדי שהמפות והטבלאות יהיו קריאות.

הסקריפט המקורי הניח שלכל צומת בגרף קיימת שורה תואמת ב-`stops.txt`, והיה קורס בהמשך עם
`KeyError` סתמי של pandas אילו לא היה כך. אנו בודקים הנחה זו במפורש ומעלים הודעת שגיאה
ניתנת לפעולה, וכן מוודאים שהקואורדינטות פורסרו כמספרים - קו רוחב שהפך ל-NaN בשקט היה מייצר
מפה שבורה ללא כל הודעת שגיאה.

In [ ]:
def attach_stop_metadata(directed, undirected, stops):
    """Copy name/code/lat/lon onto the nodes; return the active-stop table."""
    active = stops[stops["stop_id"].isin(undirected.nodes)].copy()

    missing = set(undirected.nodes) - set(active["stop_id"])
    if missing:
        raise KeyError(
            f"{len(missing)} rail stop_id(s) appear in stop_times.txt but not in stops.txt "
            f"(e.g. {sorted(missing)[:5]}). The GTFS feed files are out of sync - "
            "re-download israel-public-transportation/ and stop_times.txt from the same snapshot."
        )

    active["stop_lat"] = pd.to_numeric(active["stop_lat"], errors="coerce")
    active["stop_lon"] = pd.to_numeric(active["stop_lon"], errors="coerce")
    bad_coords = active[active[["stop_lat", "stop_lon"]].isna().any(axis=1)]
    if not bad_coords.empty:
        raise ValueError(
            f"{len(bad_coords)} rail stops have unparsable coordinates: "
            f"{bad_coords['stop_id'].tolist()[:5]}"
        )

    active = active.set_index("stop_id", drop=False)
    for stop_id in undirected.nodes:
        row = active.loc[stop_id]
        attributes = {
            "stop_name": row.get("stop_name", ""),
            "stop_code": row.get("stop_code", ""),
            "lat": float(row["stop_lat"]),
            "lon": float(row["stop_lon"]),
        }
        directed.nodes[stop_id].update(attributes)
        undirected.nodes[stop_id].update(attributes)
    return active.reset_index(drop=True)

active_stops = attach_stop_metadata(directed, undirected, stops)
print(f"metadata attached for {len(active_stops)} active rail stations")
active_stops[["stop_id", "stop_code", "stop_name", "stop_lat", "stop_lon"]].head()

## מרכזיות מדויקת, נקודות חיתוך, גשרים וקהילות

כאן הגודל הקטן של רשת הרכבת משתלם. בגרף האוטובוסים היה צורך להעריך את ה-betweenness ממדגם
של צמתי מקור; עם 67 תחנות אנו מריצים את האלגוריתם של Brandes על **כולן**, ולכן ערכי
ה-betweenness הללו מדויקים ואינם קירוב. המדדים והנימוק לכל אחד מהם:

- **degree / weighted degree** - כמה שכנים יש לתחנה, וכמה שירות מתוזמן עובר דרכה.
  weighted degree נושא את הסייג בדבר רשומות הנסיעה.
- **PageRank** (על הגרף המכוון והמשוקלל) - חשיבות תחת הילוך אקראי של נוסע.
- **betweenness** (בלתי משוקלל, בלתי מכוון) - חלקם של המסלולים הקצרים ביותר בין תחנות
  העוברים דרך תחנה נתונה. בכוונה בלתי משוקלל: אנו מעוניינים בעומס מעבר *מבני*, הנמדד
  בקפיצות, ולא בגרסה המוטה לפי תדירות השירות.
- **harmonic centrality**, מנורמל ב-`n - 1` - מדד קרבה הנשאר סופי גם אם הגרף אינו קשיר.
- **articulation points** - תחנות שהסרתן מנתקת את הגרף. אלו, ולא ציוני המרכזיות, הן התשובה
  הישירה לשאלת העמידות.
- **bridges** - קשתות שהסרתן מנתקת את הגרף.
- **קהילות Louvain** (משוקללות, עם seed) - אשכולות השירות שלוח הזמנים משרה.

כל אחד מאלה מדויק, והתא כולו רץ בהרבה פחות משנייה.

In [ ]:
def compute_metrics(directed, undirected, stop_use_counts, active_stops, seed):
    """Per-station metric table plus the bridge table. All values are exact."""
    degree = dict(undirected.degree())
    weighted_degree = dict(undirected.degree(weight="weight"))
    pagerank = nx.pagerank(directed, weight="weight")
    # normalized=True, weight=None -> exact, hop-based betweenness over all 67 sources
    betweenness = nx.betweenness_centrality(undirected, normalized=True, weight=None)
    harmonic_raw = nx.harmonic_centrality(undirected)
    harmonic_denominator = max(1, undirected.number_of_nodes() - 1)
    harmonic = {n: v / harmonic_denominator for n, v in harmonic_raw.items()}
    articulation = set(nx.articulation_points(undirected))
    bridges = set(nx.bridges(undirected))

    communities = nx.community.louvain_communities(undirected, weight="weight", seed=seed)
    community_by_node = {}
    for community_id, members in enumerate(sorted(communities, key=len, reverse=True), start=1):
        for node in members:
            community_by_node[node] = community_id

    metadata = active_stops.set_index("stop_id")
    rows = []
    for node in undirected.nodes:
        row = metadata.loc[node]
        rows.append(
            {
                "stop_id": node,
                "stop_code": row.get("stop_code", ""),
                "stop_name": row.get("stop_name", ""),
                "stop_lat": float(row["stop_lat"]),
                "stop_lon": float(row["stop_lon"]),
                "scheduled_stop_calls": int(stop_use_counts[node]),
                "degree": int(degree[node]),
                "weighted_degree": int(weighted_degree[node]),
                "in_degree": int(directed.in_degree(node)),
                "out_degree": int(directed.out_degree(node)),
                "pagerank": float(pagerank[node]),
                "betweenness": float(betweenness[node]),
                "harmonic": float(harmonic[node]),
                "is_articulation_point": node in articulation,
                "community_id": int(community_by_node[node]),
            }
        )
    metrics = pd.DataFrame(rows)

    names = metrics.set_index("stop_id")["stop_name"].to_dict()
    bridge_rows = [
        {
            "from_stop_id": source,
            "from_stop_name": names[source],
            "to_stop_id": target,
            "to_stop_name": names[target],
            "scheduled_trip_segments": int(undirected[source][target]["weight"]),
        }
        for source, target in sorted(bridges)
    ]
    return metrics, pd.DataFrame(bridge_rows)

_t0 = time.perf_counter()
metrics, bridges = compute_metrics(directed, undirected, stop_calls, active_stops, SEED)
print(f"exact metrics for {len(metrics)} stations in {time.perf_counter() - _t0:.2f}s")
metrics.nlargest(10, "betweenness")[
    ["stop_name", "degree", "weighted_degree", "betweenness", "pagerank",
     "is_articulation_point"]
]

## סטטיסטיקות של הרשת כולה

פרופיל תמציתי של הגרף: האם הוא קשיר, עד כמה הוא דליל, כמה קפיצות מפרידות בין זוג תחנות ממוצע,
וכמה נקודות חיתוך וגשרים הוא מכיל. אורך המסלול הקצר ביותר הממוצע והקוטר מחושבים על רכיב
הקשירות הגדול ביותר, שכן גדלים אלה אינם מוגדרים בין רכיבים שונים.

In [ ]:
def network_summary(graph, directed, metrics, communities):
    components = list(nx.connected_components(graph))
    largest = graph.subgraph(max(components, key=len)).copy()
    return {
        "active_stations": graph.number_of_nodes(),
        "directed_service_edges": directed.number_of_edges(),
        "undirected_service_edges": graph.number_of_edges(),
        "connected_components": len(components),
        "largest_component_stations": largest.number_of_nodes(),
        "largest_component_share": largest.number_of_nodes() / graph.number_of_nodes(),
        "average_degree": sum(dict(graph.degree()).values()) / graph.number_of_nodes(),
        "density": nx.density(graph),
        "average_clustering": nx.average_clustering(graph),
        "average_shortest_path_hops_lcc": nx.average_shortest_path_length(largest),
        "diameter_hops_lcc": nx.diameter(largest),
        "articulation_points": int(metrics["is_articulation_point"].sum()),
        "bridges": int(sum(1 for _ in nx.bridges(graph))),
        "louvain_communities": communities,
    }

n_communities = metrics["community_id"].nunique()
summary = network_summary(undirected, directed, metrics, n_communities)
for key, value in summary.items():
    print(f"{key:34s} {value:.4f}" if isinstance(value, float) else f"{key:34s} {value}")

## צמתי החיתוך וקשתות החיתוך

נקודות חיתוך וגשרים הם התוצאה המבנית החדה ביותר במחברת זו: אלו עובדות של כן/לא לגבי הגרף,
ולא ציונים הדורשים סף כלשהו. נקודת חיתוך היא תחנה שסגירתה מפצלת את גרף לוח הזמנים לחלקים
שאינם יכולים להגיע זה לזה; גשר הוא קשת בעלת אותה תכונה.

יש לשים לב לכמה מנקודות החיתוך יש betweenness *נמוך*. תחנת קצה שקטה שהיא הדרך היחידה אל
ענף מסוים היא קריטית מבחינה מבנית אף שמעט מאוד מסלולים קצרים ביותר חוצים אותה - וזו בדיוק
הסיבה לכך שמרכזיות לבדה אינה קריטריון מספק לקריטיות.

In [ ]:
articulation_points = (
    metrics[metrics["is_articulation_point"]]
    .sort_values("betweenness", ascending=False)
    [["stop_id", "stop_code", "stop_name", "degree", "weighted_degree",
      "betweenness", "community_id"]]
    .reset_index(drop=True)
)
print(f"articulation points: {len(articulation_points)}")
print(f"bridges            : {len(bridges)}")
display(articulation_points)
display(bridges.sort_values("scheduled_trip_segments", ascending=False).reset_index(drop=True))

## כמה נזק גורמת סגירה של תחנה אחת?

במקום לסמוך על ציון מרכזיות יחיד כלשהו, אנו פשוט מנסים כל תחנה: מסירים אותה, בוחנים את הנותר,
ורושמים כמה מהתחנות הנותרות נופלות אל מחוץ לרכיב הקשירות הגדול ביותר ששרד. עם 67 צמתים סריקת
כוח גס זו מיידית, והיא אמת המידה (ground truth) שדירוגי המרכזיות מנסים לקרב.

מדווחים שני שיעורים, והם עונים על שאלות שונות:
- `largest_component_share_of_remaining` - מבין התחנות שעודן פתוחות, איזה חלק עדיין יכול
  להגיע זה לזה.
- `serviceable_share_of_original` - אותה ספירה כשיעור מתוך 67 התחנות המקוריות, כלומר כמה
  מהרשת עודנה שמישה, בהתחשב גם באובדן התחנה הסגורה עצמה.

In [ ]:
def graph_state(graph, removed):
    """Connectivity of the graph after deleting the given stations."""
    remaining = set(graph.nodes).difference(removed)
    original_nodes = graph.number_of_nodes()
    if not remaining:
        return {
            "remaining_stations": 0,
            "components": 0,
            "largest_component_nodes": 0,
            "largest_component_share_of_remaining": 0.0,
            "serviceable_share_of_original": 0.0,
        }
    subgraph = graph.subgraph(remaining)
    component_sizes = [len(c) for c in nx.connected_components(subgraph)]
    largest = max(component_sizes)
    return {
        "remaining_stations": len(remaining),
        "components": len(component_sizes),
        "largest_component_nodes": largest,
        "largest_component_share_of_remaining": largest / len(remaining),
        "serviceable_share_of_original": largest / original_nodes,
    }

def single_station_damage(graph, metrics):
    """Exhaustively close each station in turn and measure the fragmentation."""
    names = metrics.set_index("stop_id")["stop_name"].to_dict()
    rows = []
    for node in graph.nodes:
        state = graph_state(graph, [node])
        rows.append(
            {
                "stop_id": node,
                "stop_name": names[node],
                **state,
                "stations_outside_largest_component": int(
                    state["remaining_stations"] - state["largest_component_nodes"]
                ),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["stations_outside_largest_component", "components", "stop_name"],
        ascending=[False, False, True],
    )

damage = single_station_damage(undirected, metrics)
worst = damage.iloc[0]
print(
    f"Worst single closure: {worst['stop_name']} -> "
    f"{worst['components']} components, only "
    f"{100 * worst['serviceable_share_of_original']:.1f}% of stations still connected"
)
damage.head(12)[
    ["stop_name", "components", "largest_component_nodes",
     "stations_outside_largest_component", "serviceable_share_of_original"]
]

## סגירות ממוקדות מול כשלים אקראיים

ניסוי החוסן הקלאסי. אנו מסירים תחנות אחת-אחת בחמישה סדרים ממוקדים שונים (degree הגבוה ביותר,
weighted degree, PageRank, betweenness, וסדר "articulation first" הנותן עדיפות לצמתי חיתוך
ולאחר מכן לנזק הנצפה מסגירת תחנה בודדת) ועוקבים אחר גודלו של רכיב הקשירות הגדול ביותר. כקו
בסיס אנו חוזרים על אותן הסרות ב-`RANDOM_TRIALS` סדרים אקראיים וממצעים את התוצאה.

הדירוגים סטטיים: הם מחושבים פעם אחת על הגרף השלם ולאחר מכן נעקבים, מה שמדמה תוקף המתכנן
מתוך לוח זמנים מפורסם ולא תוקף המחשב מחדש מרכזיות לאחר כל סגירה. חישוב מחדש היה מזיק יותר
בהכרח, ולכן העקומות הממוקדות שלהלן הן חסם תחתון שמרני על המקרה הגרוע ביותר.

**עלות:** `RANDOM_TRIALS x (MAX_REMOVALS + 1)` הערכות קשירות, כמה שניות בערכי ברירת המחדל
של 500 כפול 16.

In [ ]:
def resilience_curves(graph, metrics, damage, max_removals, random_trials, seed):
    damage_rank = damage.set_index("stop_id")["stations_outside_largest_component"]
    ranked = metrics.copy()
    ranked["single_station_damage"] = ranked["stop_id"].map(damage_rank)
    rankings = {
        "degree": ranked.sort_values(
            ["degree", "weighted_degree"], ascending=False
        )["stop_id"].tolist(),
        "weighted_degree": ranked.sort_values("weighted_degree", ascending=False)[
            "stop_id"
        ].tolist(),
        "pagerank": ranked.sort_values("pagerank", ascending=False)["stop_id"].tolist(),
        "betweenness": ranked.sort_values("betweenness", ascending=False)[
            "stop_id"
        ].tolist(),
        "articulation_priority": ranked.sort_values(
            ["is_articulation_point", "single_station_damage", "degree"],
            ascending=False,
        )["stop_id"].tolist(),
    }
    max_removals = min(max_removals, graph.number_of_nodes() - 1)

    rows = []
    for strategy, ranking in rankings.items():
        for removed_count in range(max_removals + 1):
            rows.append(
                {
                    "strategy": strategy,
                    "removed_stations": removed_count,
                    **graph_state(graph, ranking[:removed_count]),
                }
            )

    rng = random.Random(seed)  # seeded so the baseline is reproducible
    nodes = list(graph.nodes)
    random_states = defaultdict(list)
    for _ in range(random_trials):
        ordering = rng.sample(nodes, len(nodes))
        for removed_count in range(max_removals + 1):
            random_states[removed_count].append(graph_state(graph, ordering[:removed_count]))
    for removed_count, states in random_states.items():
        row = {"strategy": "random_mean", "removed_stations": removed_count}
        for field in states[0]:
            row[field] = float(np.mean([float(state[field]) for state in states]))
        rows.append(row)
    return pd.DataFrame(rows)

_t0 = time.perf_counter()
resilience = resilience_curves(
    undirected, metrics, damage, MAX_REMOVALS, RANDOM_TRIALS, SEED
)
print(f"resilience curves computed in {time.perf_counter() - _t0:.1f}s")
resilience.pivot(
    index="removed_stations", columns="strategy", values="serviceable_share_of_original"
).round(3)

## תחנה אחת מול הטלת מטבע אחת

המספר הברור ביותר במחברת זו: מה קורה לאחר סגירה אחת בדיוק, כאשר היא נבחרת בידי יריב לעומת
בחירה אקראית.

In [ ]:
worst_row = damage.iloc[0]
random_one = resilience[
    (resilience["strategy"] == "random_mean") & (resilience["removed_stations"] == 1)
].iloc[0]

targeted_pct = 100 * float(worst_row["serviceable_share_of_original"])
random_pct = 100 * float(random_one["serviceable_share_of_original"])
print(f"Closing the single worst station ({worst_row['stop_name']}):")
print(f"  largest connected component = {targeted_pct:.1f}% of the original network")
print(f"  network splits into {int(worst_row['components'])} components")
print(f"Closing one station at random (mean of {RANDOM_TRIALS} trials):")
print(f"  largest connected component = {random_pct:.1f}% of the original network")
print(f"Gap: {random_pct - targeted_pct:.1f} percentage points from one informed choice.")

## איורים

חמישה איורים, כל אחד עונה על שאלה אחת:

1. **מפת הרשת** - הגרף משורטט על קואורדינטות אמיתיות, צבע הצומת = betweenness מדויק,
   גודל הצומת = weighted degree. כאן צורת השדרה-והענפים נעשית ברורה.
2. **תחנות ה-betweenness המובילות** - מי נושא בעומס המסלולים הקצרים ביותר.
3. **נזק מתחנה בודדת** - אילו סגירות אכן מפצלות את הרשת (מוצגות רק תחנות המנתקות לפחות תחנה
   אחת נוספת; רוב התחנות אינן מנתקות אף אחת).
4. **עקומות עמידות** - אסטרטגיות ממוקדות מול קו הבסיס האקראי.
5. **מתאמי מרכזיות** - מתאם Spearman בין המדדים, כדי לבדוק אם הם מודדים את אותו הדבר או
   דברים שונים באמת.

האיורים נכתבים לתיקיית `figures/` של השלב וגם מוצגים בתוך המחברת.

In [ ]:
def plot_network_map(graph, metrics, path):
    indexed = metrics.set_index("stop_id")
    positions = {
        node: (float(indexed.loc[node, "stop_lon"]), float(indexed.loc[node, "stop_lat"]))
        for node in graph.nodes
    }
    segments = [[positions[s], positions[t]] for s, t in graph.edges]
    fig, axis = plt.subplots(figsize=(8, 10))
    axis.add_collection(
        LineCollection(segments, colors="#94a3b8", linewidths=0.8, alpha=0.5)
    )
    node_list = list(graph.nodes)
    values = indexed.loc[node_list, "betweenness"].to_numpy(float)
    weights = indexed.loc[node_list, "weighted_degree"].to_numpy(float)
    sizes = 22 + 180 * np.sqrt(weights / max(weights.max(), 1))
    scatter = axis.scatter(
        [positions[n][0] for n in node_list],
        [positions[n][1] for n in node_list],
        c=values, s=sizes, cmap="magma_r",
        edgecolor="white", linewidth=0.5, zorder=3,
    )
    label_offsets = [(8, 12), (8, -13), (-8, 12), (8, -13), (-8, 12), (8, -13), (8, 12)]
    for node, offset in zip(indexed.nlargest(7, "betweenness").index, label_offsets):
        x_pos, y_pos = positions[node]
        axis.annotate(
            indexed.loc[node, "stop_name"],  # raw Hebrew: the bidi patch fixes it
            (x_pos, y_pos), xytext=offset, textcoords="offset points", fontsize=7,
            ha="left" if offset[0] > 0 else "right",
            arrowprops={"arrowstyle": "-", "color": "#64748b", "lw": 0.5},
        )
    fig.colorbar(scatter, ax=axis, label="Betweenness centrality (exact)")
    axis.set_title("Israel Railways scheduled-service network")
    axis.set_xlabel("Longitude")
    axis.set_ylabel("Latitude")
    axis.autoscale()
    fig.tight_layout()
    fig.savefig(path, dpi=220)
    plt.show()

plot_network_map(undirected, metrics, FIGURES / "rail_network_map.png")

### דירוג betweenness ונזק מתחנה בודדת

תרשים העמודות הראשון מדרג תחנות לפי betweenness מדויק; השני מציג את תוצאת הנזק שהתקבלה בכוח
גס. ההשוואה ביניהם היא העיקר - הסדר דומה בראש הרשימה אך אינו זהה, ויש תחנות בעלות betweenness
גבוה שאינן מפצלות דבר, משום שהרשת עוקפת אותן.

In [ ]:
def plot_top_stations(metrics, path):
    top = metrics.nlargest(12, "betweenness").sort_values("betweenness")
    fig, axis = plt.subplots(figsize=(9, 6))
    axis.barh(list(top["stop_name"]), top["betweenness"], color="#2563eb")
    axis.set_xlabel("Betweenness centrality (exact)")
    axis.set_title("Rail stations most often bridging shortest service paths")
    axis.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()

def plot_single_station_damage(damage, path):
    top = damage[damage["stations_outside_largest_component"] > 0].head(15)
    top = top.sort_values("stations_outside_largest_component")
    fig, axis = plt.subplots(figsize=(9, 6))
    axis.barh(
        list(top["stop_name"]),
        top["stations_outside_largest_component"],
        color="#dc2626",
    )
    axis.set_xlabel("Other stations separated from the largest component")
    axis.set_title("Damage caused by closing one rail station")
    axis.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()

plot_top_stations(metrics, FIGURES / "top_betweenness_stations.png")
plot_single_station_damage(damage, FIGURES / "single_station_damage.png")

### עקומות עמידות והסכמה בין המדדים

תרשים העמידות מציג את רכיב הקשירות הגדול ביותר כאחוז מ-67 התחנות המקוריות לאחר כל הסרה;
הקו האפור המקווקו הוא ממוצע הכשלים האקראיים. מפת החום בודקת לאחר מכן עד כמה חמשת מדדי
המרכזיות שלנו עודפים זה לזה: מדדים בעלי מתאם חזק מספרים לנו את אותו הסיפור, וכל מתאם חלש
מסמן תפיסת חשיבות שונה באמת.

In [ ]:
def plot_resilience(resilience, path):
    colors = {
        "degree": "#0f766e",
        "weighted_degree": "#2563eb",
        "pagerank": "#7c3aed",
        "betweenness": "#dc2626",
        "articulation_priority": "#ea580c",
        "random_mean": "#64748b",
    }
    fig, axis = plt.subplots(figsize=(9, 6))
    for strategy, group in resilience.groupby("strategy", sort=False):
        axis.plot(
            group["removed_stations"],
            100 * group["serviceable_share_of_original"],
            marker="o" if strategy != "random_mean" else None,
            markersize=3,
            linewidth=2,
            linestyle="--" if strategy == "random_mean" else "-",
            color=colors.get(strategy),
            label=strategy.replace("_", " "),
        )
    axis.set_xlabel("Stations removed")
    axis.set_ylabel("Largest connected component (% of original stations)")
    axis.set_title("Rail-network resilience: targeted closures vs random failures")
    axis.set_ylim(0, 102)
    axis.grid(alpha=0.2)
    axis.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()

def plot_centrality_correlation(metrics, path):
    columns = ["degree", "weighted_degree", "pagerank", "betweenness", "harmonic"]
    correlation = metrics[columns].corr(method="spearman")
    fig, axis = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        correlation, annot=True, fmt=".2f", cmap="vlag",
        vmin=-1, vmax=1, square=True, ax=axis,
    )
    axis.set_title("Spearman correlation between rail centrality metrics")
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()

plot_resilience(resilience, FIGURES / "rail_resilience_curve.png")
plot_centrality_correlation(metrics, FIGURES / "centrality_correlation.png")

## שמירת התוצאות

כל טבלה וכן סיכום ה-JSON נכתבים לתיקיית השלב של מחברת זו
(`outputs/nb/11_rail_network_analysis/`), ומשאירים את התוצאות המצוטטות בדוח שב-`outputs/rail/`
ללא שינוי. קובצי ה-CSV משתמשים ב-`utf-8-sig` כדי ש-Excel יפתח נכון את שמות התחנות בעברית.
הסיכום מתעד גם את ספירות הבנייה (שורות שנסרקו, שורות שנשמרו, נסיעות, חלון השירות), כך שניתן
לעקוב מהגרף בחזרה אל תצלום המצב של הקובץ שממנו נוצר.

In [ ]:
inventory.to_csv(TABLES / "mode_inventory.csv", index=False, encoding="utf-8-sig")
metrics.sort_values("betweenness", ascending=False).to_csv(
    TABLES / "rail_station_metrics.csv", index=False, encoding="utf-8-sig"
)
metrics.nlargest(15, "betweenness").to_csv(
    TABLES / "top_betweenness_stations.csv", index=False, encoding="utf-8-sig"
)
metrics.nlargest(15, "weighted_degree").to_csv(
    TABLES / "top_weighted_degree_stations.csv", index=False, encoding="utf-8-sig"
)
articulation_points.to_csv(
    TABLES / "rail_articulation_points.csv", index=False, encoding="utf-8-sig"
)
bridges.to_csv(TABLES / "rail_bridges.csv", index=False, encoding="utf-8-sig")
damage.to_csv(TABLES / "single_station_damage.csv", index=False, encoding="utf-8-sig")
resilience.to_csv(
    TABLES / "rail_resilience_curves.csv", index=False, encoding="utf-8-sig"
)

edge_rows = [
    {
        "from_stop_id": source,
        "to_stop_id": target,
        "scheduled_trip_segments": int(data["weight"]),
    }
    for source, target, data in directed.edges(data=True)
]
pd.DataFrame(edge_rows).sort_values("scheduled_trip_segments", ascending=False).to_csv(
    TABLES / "rail_service_edges.csv", index=False, encoding="utf-8-sig"
)

build_summary = {
    "source_stop_time_rows": int(source_rows),
    "rail_stop_time_rows": int(rail_rows),
    "rail_route_records": int(len(rail_routes)),
    "unique_route_descriptions": int(rail_routes["route_long_name"].nunique()),
    "scheduled_rail_trips": int(len(rail_trips)),
    "active_rail_stations": int(undirected.number_of_nodes()),
    "directed_service_edges": int(directed.number_of_edges()),
    "undirected_service_edges": int(undirected.number_of_edges()),
    "service_start_date": start_date,
    "service_end_date": end_date,
}
with (STAGE / "summary.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {"build": build_summary, "network": summary},
        handle, ensure_ascii=False, indent=2,
    )

print("tables written to :", TABLES)
print("figures written to:", FIGURES)
print("summary written to:", STAGE / "summary.json")

## מסקנות

**רשת הרכבת היא שדרה דלילה, וזה מה שהופך אותה לשברירית.** 67 תחנות פעילות ו-99 קשתות שירות
בלתי מכוונות נותנות degree ממוצע של כ-3.0 וצפיפות של כ-0.045. הגרף קשיר לחלוטין, אך הוא קשיר
באופן שבו עץ עם מעט מעגלים קשיר: זוג תחנות ממוצע מרוחק בכ-5.6 קפיצות והקוטר הוא 13 קפיצות.

**14 תחנות הן נקודות חיתוך ו-11 קשתות הן גשרים.** מדובר בכתחנה אחת מכל חמש שסגירתה מנתקת חלק
משירות הרכבת הארצי, ובקשת אחת מכל תשע בעלת אותה תכונה. זוהי עובדה מבנית לגבי גרף לוח הזמנים,
ולא ציון עם סף, וזהו ממצא העמידות החזק ביותר כאן.

**סגירה אחת מושכלת שקולה לעשרות סגירות אקראיות.** הסרת תל אביב סבידור מרכז לבדה מפצלת את
הרשת לשניים ומותירה רק כ-**49%** מהתחנות מקושרות הדדית. הסרת תחנה אחת אקראית מותירה כ-**96%**.
פער של כ-47 נקודות אחוז מבחירה בודדת הוא חתימתה של רשת עם מעט מאוד מסלולים עודפים - וזה נכון
עבור ארבע תחנות תל אביב באופן כללי, שכן למעשה כל שירות הרכבת בציר צפון-דרום מנותב דרך אותו
מסדרון.

**מרכזיות היא פרוקסי טוב אך לא מושלם לקריטיות.** מדדי המרכזיות מתואמים חזק זה עם זה, וראש
דירוג ה-betweenness אכן חופף לראש דירוג הנזק. אולם לכמה מנקודות החיתוך יש betweenness בלתי
מרשים - תחנת קצה שקטה של ענף היא קריטית עבור קומץ התחנות שמאחוריה ועבור אף אחד אחר. סריקת
הכוח הגס של תחנה בודדת, שאפשרית רק משום שהגרף קטן, היא התשובה הכנה; ברשת האוטובוסים איננו
יכולים להרשות אותה לעצמנו ועלינו לסמוך על הפרוקסים.

**מגבלות - אין לקרוא במספרים אלה יותר מהמידה.**
1. *שירות מתוזמן, לא מסילה פיזית.* קשתות מחברות תחנות שרכבות עוצרות בהן ברצף. תבניות אקספרס
   מקשרות אפוא תחנות שאינן סמוכות פיזית, ו"גשר" כאן פירושו קשת שירות בלתי ניתנת להחלפה בלוח
   הזמנים המפורסם, ולא בהכרח מקטע מסילה יחיד. מחקר תשתית אמיתי היה זקוק לנתוני מסילות.
2. *המשקלים סופרים רשומות נסיעה, לא ימי שירות.* תבנית שירות הפועלת רק בימי ראשון תורמת אותו
   משקל לנסיעה כמו תבנית יומיומית, ולכן weighted degree ו-PageRank מודדים עושר של תבניות
   שירות ולא נוסעים או רכבות לשבוע. כל התוצאות המבניות שלעיל משתמשות בגרף הבלתי משוקלל
   וחסינות מפני כך.
3. *תצלום מצב יחיד של הקובץ.* כל האמור מתאר את חלון לוח הזמנים היחיד שהודפס בתא חלון השירות
   שלעיל. עבודות תחזוקה, לוחות זמנים עונתיים או קו חדש היו משנים את הטופולוגיה.
4. *טופולוגיה בלבד.* הקשירות נמדדת בתחנות, לא בנוסעים. אובדן ענף בן שתי תחנות ואובדן מקטע בן
   שתי תחנות במסדרון תל אביב נספרים כאן באופן זהה; במציאות אין הדבר כך. שקלול לפי היקף נסיעות
   היה מחדד כל מסקנה.
5. *סדר תקיפה סטטי.* העקומות הממוקדות עוקבות אחר דירוג שחושב על הגרף השלם. יריב המחשב מחדש
   לאחר כל סגירה היה גורם נזק רב יותר, ולכן עקומות אלה הן חסם תחתון על המקרה הגרוע ביותר.